In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [29]:
import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


Using device: cuda


In [95]:
BASE_PATH = "/kaggle/input/461054610546105"

# ✅ Use the ACTUAL filename
labels = pd.read_parquet(f"{BASE_PATH}/labels.parqet")

# Small subset for Tier-2 experiments
labels = labels.sample(1000, random_state=42).reset_index(drop=True)

# Convert relative paths → full paths
labels["path"] = BASE_PATH + "/" + labels["path"]

labels.head()



,path,sign,Usage
0,/kaggle/input/461054610546105/test_landmark_fi...,find,Private
1,/kaggle/input/461054610546105/test_landmark_fi...,outside,Public
2,/kaggle/input/461054610546105/test_landmark_fi...,read,Private
3,/kaggle/input/461054610546105/test_landmark_fi...,hear,Private
4,/kaggle/input/461054610546105/test_landmark_fi...,shirt,Private


In [96]:
tier2_df = labels.copy()

tier2_labels = sorted(tier2_df["sign"].unique())
label_map = {label: i for i, label in enumerate(tier2_labels)}
NUM_CLASSES = len(label_map)

print("Tier-2 Classes:", NUM_CLASSES)


Tier-2 Classes: 246


In [116]:
def extract_mock_gru_input(parquet_path, window=5, max_len=1500):
    df = pd.read_parquet(parquet_path)

    df = df[["frame", "type", "landmark_index", "x", "y"]]
    df = df.dropna()

    # Create unique landmark id like "face_0", "left_hand_5"
    df["landmark_id"] = df["type"].astype(str) + "_" + df["landmark_index"].astype(str)

    # Pivot safely
    x_pivot = df.pivot_table(index="frame", columns="landmark_id", values="x")
    y_pivot = df.pivot_table(index="frame", columns="landmark_id", values="y")

    # Align frames and landmarks
    x_pivot, y_pivot = x_pivot.align(y_pivot, join="inner", axis=0)

    if len(x_pivot) < window + 1:
        return None

    x = x_pivot.to_numpy(dtype=np.float32)
    y = y_pivot.to_numpy(dtype=np.float32)

    x = np.nan_to_num(x)
    y = np.nan_to_num(y)

    dx = np.diff(x, axis=0)
    dy = np.diff(y, axis=0)
    speed = np.sqrt(dx**2 + dy**2)

    feats = []
    for i in range(len(dx) - window + 1):
        fx = dx[i:i+window]
        fy = dy[i:i+window]
        fs = speed[i:i+window]

        feats.append([
            fx.mean(),
            fy.mean(),
            fs.mean(),
            fx.var(),
            fy.var(),
            fs[-1].mean() - fs[0].mean()
        ])

    if len(feats) == 0:
        return None

    seq = np.array(feats, dtype=np.float32)
    seq = (seq - seq.mean(0)) / (seq.std(0) + 1e-6)

    return torch.tensor(seq[:max_len])


In [98]:
class SignDataset(Dataset):
    def __init__(self, labels_df, label_map):
        self.labels = labels_df.reset_index(drop=True)
        self.label_map = label_map

    def __getitem__(self, idx):
        row = self.labels.iloc[idx]
        x = extract_mock_gru_input(row["path"])

        if x is None:
            return None

        y = self.label_map[row["sign"]]
        return x, torch.tensor(y), x.shape[0]

    def __len__(self):
        return len(self.labels)



In [100]:
def collate_fn(batch):
    batch = [b for b in batch if b is not None]
    if len(batch) == 0:
        return None

    xs, ys, lengths = zip(*batch)
    xs_padded = pad_sequence(xs, batch_first=True)

    return xs_padded, torch.tensor(ys), torch.tensor(lengths)


In [102]:
dataset = SignDataset(tier2_df, label_map)

loader = DataLoader(
    dataset,
    batch_size=16,
    shuffle=True,
    collate_fn=collate_fn,
    num_workers=2,
    pin_memory=True
)


In [103]:
class GRUClassifier(nn.Module):
    def __init__(self, input_dim=6, hidden=128, classes=10):
        super().__init__()
        self.gru = nn.GRU(input_dim, hidden, batch_first=True)
        self.fc = nn.Linear(hidden, classes)

    def forward(self, x, lengths):
        packed = nn.utils.rnn.pack_padded_sequence(
            x, lengths.cpu(), batch_first=True, enforce_sorted=False
        )
        _, h = self.gru(packed)
        return self.fc(h[-1])



In [104]:
class GRUClassifier(nn.Module):
    def __init__(self, input_dim=6, hidden=128, classes=10):
        super().__init__()
        self.gru = nn.GRU(input_dim, hidden, batch_first=True)
        self.fc = nn.Linear(hidden, classes)

    def forward(self, x, lengths):
        packed = nn.utils.rnn.pack_padded_sequence(
            x, lengths.cpu(), batch_first=True, enforce_sorted=False
        )
        _, h = self.gru(packed)
        return self.fc(h[-1])


In [110]:
device = "cuda" if torch.cuda.is_available() else "cpu"

model = GRUClassifier(classes=NUM_CLASSES).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)


In [112]:
valid = 0
for i in range(50):
    sample = extract_mock_gru_input(tier2_df.iloc[i]["path"])
    if sample is not None:
        valid += 1

print("Valid samples in first 50:", valid)


Valid samples in first 50: 0


In [113]:
sample_path = tier2_df.iloc[0]["path"]
df = pd.read_parquet(sample_path)

print("Total columns:", len(df.columns))
print(df.columns[:30])  # show first 30


Total columns: 7
Index(['frame', 'row_id', 'type', 'landmark_index', 'x', 'y', 'z'], dtype='object')


In [117]:
valid = 0
for i in range(50):
    sample = extract_mock_gru_input(tier2_df.iloc[i]["path"])
    if sample is not None:
        valid += 1

print("Valid samples in first 50:", valid)


Valid samples in first 50: 49


In [118]:
for epoch in range(5):
    model.train()
    total_loss, correct, total = 0, 0, 0

    for batch in tqdm(loader):
        if batch is None:
            continue

        x, y, lengths = batch
        x, y = x.to(device), y.to(device)

        optimizer.zero_grad()
        logits = model(x, lengths)
        loss = criterion(logits, y)

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
        optimizer.step()

        total_loss += loss.item()
        correct += (logits.argmax(1) == y).sum().item()
        total += y.size(0)

    print(f"Epoch {epoch+1} | Loss {total_loss/total:.4f} | Acc {correct/total:.4f}")


100%|██████████| 63/63 [00:20<00:00,  3.07it/s]


Epoch 1 | Loss 0.3492 | Acc 0.0060


100%|██████████| 63/63 [00:20<00:00,  3.09it/s]


Epoch 2 | Loss 0.3419 | Acc 0.0191


100%|██████████| 63/63 [00:20<00:00,  3.09it/s]


Epoch 3 | Loss 0.3329 | Acc 0.0181


100%|██████████| 63/63 [00:20<00:00,  3.02it/s]


Epoch 4 | Loss 0.3226 | Acc 0.0231


100%|██████████| 63/63 [00:20<00:00,  3.09it/s]

Epoch 5 | Loss 0.3130 | Acc 0.0302
